# 02 — Generate 100,000 users

Builds the **client** rows of `synth_users` (ids `1..N_USERS`, `is_handyman = false`) with exactly the columns of `maintops.users`. `03_generate_handymen` appends the handyman accounts (ids `N_USERS+1..`).

* **Address / coordinates**: a unique real address from `synth_address_pool` (`purpose = 'user'`).
* **Email**: synthesised from the person's name (`anna.mueller@example.com`, `j.schmidt42@example.org` …). Unique, lowercase, and only on reserved `example.*` domains so nothing can reach a real mailbox. Users log in with it.
* **Password**: the demo password from `00_config`, hashed with Argon2id (one hash reused for all rows, so generation stays fast).

In [1]:
%pip install argon2-cffi

Note: you may need to restart the kernel to use updated packages.


In [2]:
try:
    dbutils.library.restartPython()
except NameError:  # running locally: no dbutils, nothing to restart
    pass

In [3]:
# Environment bootstrap: runs unchanged on Databricks and on a local machine.
# Locally it connects through Databricks Connect (serverless unless DATABRICKS_CLUSTER_ID is set)
# and provides `spark`, `display` and `list_volume_files`. Override the profile with DATABRICKS_CONFIG_PROFILE.
import os
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
LOCAL_SPARK_DEPENDENCIES = []   # pip packages the remote workers need (serverless Databricks Connect only)
if not IS_DATABRICKS:
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "george.sokolovsky2001@gmail.com")
    from databricks.connect import DatabricksSession

    _builder = DatabricksSession.builder
    if os.environ.get("DATABRICKS_CLUSTER_ID"):
        _builder = _builder.clusterId(os.environ["DATABRICKS_CLUSTER_ID"])
    else:
        _builder = _builder.serverless()
        if LOCAL_SPARK_DEPENDENCIES:
            from databricks.connect import DatabricksEnv
            _builder = _builder.withEnvironment(DatabricksEnv().withDependencies(LOCAL_SPARK_DEPENDENCIES))
    spark = _builder.getOrCreate()

    def display(obj, n=1000):
        from IPython.display import display as _display
        _display(obj.limit(n).toPandas() if hasattr(obj, "toPandas") else obj)

def list_volume_files(directory, suffix=""):
    """File names in a Unity Catalog Volume folder (the Volume is only mounted on Databricks, so go through the Files API locally)."""
    if IS_DATABRICKS:
        return [f for f in os.listdir(directory) if f.endswith(suffix)] if os.path.isdir(directory) else []
    from databricks.sdk import WorkspaceClient
    return [e.name for e in WorkspaceClient().files.list_directory_contents(directory) if e.name.endswith(suffix)]


In [4]:
%run ./00_config

In [5]:
# Cell 1 — Load the user addresses
import numpy as np, pandas as pd
from argon2 import PasswordHasher

addr = (spark.table(T_ADDRESS_POOL).where("purpose = 'user'").orderBy("address_id").toPandas())
assert len(addr) == N_USERS, f"expected {N_USERS} user addresses, found {len(addr)} — re-run 01_real_addresses"

In [6]:
# Cell 2 — Build the users
rng = np.random.default_rng(SEED + 1)
password_hash = PasswordHasher().hash(DEMO_PASSWORD)

ids = np.arange(1, N_USERS + 1, dtype="int64")
dob = pd.to_datetime("1950-01-01") + pd.to_timedelta(rng.integers(0, (pd.Timestamp("2005-12-31") - pd.Timestamp("1950-01-01")).days, N_USERS), unit="D")

first_names = rng.choice(FIRST_NAMES, N_USERS)
last_names  = rng.choice(LAST_NAMES, N_USERS)
used_emails = set()
emails      = [make_email(rng, f, l, used_emails) for f, l in zip(first_names, last_names)]

users = pd.DataFrame({
    "id":            ids,
    "email":         emails,
    "password_hash": password_hash,
    "first_name":    first_names,
    "last_name":     last_names,
    "date_of_birth": dob.date,
    "phone":         [make_phone(rng) for _ in range(N_USERS)],
    "house":         (addr["street"] + " " + addr["housenumber"]).str.slice(0, 128).values,
    "postal_code":   addr["postcode"].values,
    "city":          addr["city"].values,
    "state":         addr["state"].values,
    "country":       "Germany",
    "latitude":      addr["latitude"].values,
    "longitude":     addr["longitude"].values,
    "is_handyman":   False,
    "created_at":    pd.to_datetime(ACCOUNTS_START) + pd.to_timedelta(rng.integers(0, int((pd.Timestamp(ACCOUNTS_END) - pd.Timestamp(ACCOUNTS_START)).total_seconds()), N_USERS), unit="s"),
    "is_active":     rng.random(N_USERS) < CLIENT_ACTIVE_SHARE,
})
assert users["email"].is_unique and users["email"].str.len().max() <= 64 and not users.duplicated(subset=["house", "postal_code", "city"]).any()

In [7]:
# Cell 3 — Save to Delta
(spark.createDataFrame(users)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(T_USERS))
print(f"Saved {N_USERS:,} users to {T_USERS}")
display(spark.table(T_USERS).limit(10))

Saved 100,000 users to bootcamp_students.maintops.synth_users


,id,email,password_hash,first_name,last_name,date_of_birth,phone,house,postal_code,city,state,country,latitude,longitude
0,50001,matthiaslorenz@example.net,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Matthias,Lorenz,1958-02-18,+49 170 5540487,Im Hof 15,66557,Illingen,Saarland,Germany,49.386590,7.040961
1,50002,ralf.schmidt72@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Ralf,Schmidt,1995-12-29,+49 160 5275572,Wendlohstraße 130a,22459,Hamburg,Hamburg,Germany,53.630503,9.938203
2,50003,claudiathomas@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Claudia,Thomas,1970-04-10,+49 152 8781569,Wientapperweg 16g,22589,Hamburg,Hamburg,Germany,53.571610,9.831884
3,50004,franziska_arslan2@example.org,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Franziska,Arslan,1985-03-29,+49 157 1379285,Im Mullsen 78,21149,Hamburg,Hamburg,Germany,53.474532,9.831448
4,50005,h.richter17@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Hannah,Richter,1962-03-27,+49 162 7743811,Wildschwanbrook 75,22145,Hamburg,Hamburg,Germany,53.629907,10.174070
5,50006,frank-bohm6@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Frank,Bohm,1998-06-25,+49 157 1288080,Schulenburgring 64,21031,Hamburg,Hamburg,Germany,53.508898,10.191182
6,50007,martin.aylin4@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Aylin,Martin,1983-06-14,+49 151 9031301,Lindenallee 10a,15738,Zeuthen,Berlin,Germany,52.344024,13.633523
7,50008,marek.popov49@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Marek,Popov,2004-05-02,+49 152 4417686,Hugenottenstraße 139,66333,Völklingen,Saarland,Germany,49.217155,6.794017
8,50009,oliver.keller63@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Oliver,Keller,1971-09-27,+49 157 6162541,Hohweg 1a,28219,Bremen,Bremen,Germany,53.105006,8.795371
9,50010,m.yilmaz91@example.com,"$argon2id$v=19$m=65536,t=3,p=4$NRXlzx/6uqlNdaz...",Murat,Yilmaz,2002-05-08,+49 160 1260816,Primsstraße 10,66763,Dillingen/Saar,Saarland,Germany,49.368606,6.755930
